# Workspace — [담당자명] [담당 변수 그룹]

**이 노트북은 깃에 안 올라감 (`.gitignore` 처리됨).** 자유롭게 사용.

작업 흐름:
1. 이 템플릿을 복사: `cp _template.ipynb 본인이름_그룹.ipynb`
2. 외부 데이터 로드 → 분석 → 변수 생성 코드 작성
3. 코드가 정리되면 `src/features.py`에 함수로 옮김
4. 채택/탈락 판단 + 근거 메모 (자기 작업 노트에)
5. PR로 `src/features.py` 변경분만 머지

## 1. 라이브러리 로드

In [103]:
import os
import sys
import itertools
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score

import platform
if platform.system() == 'Darwin':
    matplotlib.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    matplotlib.rcParams['font.family'] = 'Malgun Gothic'
else:
    matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

sys.path.append('../../src')
from features import add_features_unique  # 공용 헬퍼

## 2. 데이터 로드

In [ ]:
df = pd.read_csv('../data/raw/apt_preprocessed.csv', encoding='utf-8')
school = pd.read_csv('../data/raw/서울학교.csv', encoding='utf-8')
park = pd.read_csv('../data/raw/서울공원.csv', encoding='cp949')
academy = pd.read_csv('../data/raw/서울_학원.csv', header=None, encoding='cp949')[[1, 31, 38, 37]]
hospital = pd.read_csv('../data/raw/서울_의료시설.csv', header=None, encoding='cp949')[[1, 31, 38, 37]]

academy.columns = ['시설명', '주소', '위도', '경도']
hospital.columns = ['시설명', '주소', '위도', '경도']

for name, d in [('apt', df), ('school', school), ('park', park),
                ('academy', academy), ('hospital', hospital), ('mart', mart)]:
    print(f'{name:>12}: {d.shape}')

         apt: (173399, 15)
      school: (1314, 7)
        park: (1761, 19)
     academy: (45080, 4)
    hospital: (18999, 4)
        mart: (712, 26)


## 3. 외부 데이터 탐색

컬럼 구조 확인, 위도/경도 컬럼명 파악, 결측치/이상치 점검.

In [ ]:
RADIUS_LIMIT = 0.01  

def count_infrastructure_nearby(apt_df, infra_df):
    if infra_df.empty or '위도' not in infra_df.columns:
        return np.zeros(len(apt_df))
        
    infra_coords = infra_df[['위도', '경도']].values
    tree = cKDTree(infra_coords)
    
    apt_coords = apt_df[['위도', '경도']].values
    counts = [len(idx) for idx in tree.query_ball_point(apt_coords, r=RADIUS_LIMIT)]
    return counts

df['주변학교수'] = count_infrastructure_nearby(df, school)
df['주변공원수'] = count_infrastructure_nearby(df, park)
df['주변학원수'] = count_infrastructure_nearby(df, academy)
df['주변병원수'] = count_infrastructure_nearby(df, hospital)

print(f'최종 데이터셋 크기: {df.shape}')

최종 데이터셋 크기: (173399, 19)


## 4. 변수 생성 테스트

공용 헬퍼 함수로 빠르게 거리/카운트 변수 만들어 분석.

[3개 조합] ['주변공원수', '주변학원수', '주변병원수']  정확도 : 0.8598

In [ ]:
infra_features = ['주변학교수', '주변공원수', '주변학원수', '주변병원수']

possible_base = ['전용면적', '층', '건축년도', '건물면적', '층정보']
base_features = [col for col in possible_base if col in df.columns]

target_col = None
for col in ['실거래가', '물건금액', '물건금액(만원)', '가격', '거래금액', 'price', '보증금액']:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    potential_cols = [c for c in df.columns if '금액' in c or '가' in c or 'price' in c.lower()]
    if potential_cols:
        target_col = potential_cols[0]
    else:
        target_col = df.columns[-1]

y = df[target_col]
results = []

for k in range(1, len(infra_features) + 1):
    for comb in itertools.combinations(infra_features, k):
        selected_infra = list(comb)
        
        X_features = base_features + selected_infra
        X = df[X_features]
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        model = RandomForestRegressor(n_estimators=20, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        score = r2_score(y_test, y_pred)
        
        results.append({
            '개수': k,
            '변수조합': selected_infra,
            'R2_Score': score
        })
        
        print(f"예측 정확도: {selected_infra} | {score:.4f}")

예측 정확도: ['주변학교수'] | 0.4310
예측 정확도: ['주변공원수'] | 0.4961
예측 정확도: ['주변학원수'] | 0.8170
예측 정확도: ['주변병원수'] | 0.7731
예측 정확도: ['주변학교수', '주변공원수'] | 0.7695
예측 정확도: ['주변학교수', '주변학원수'] | 0.8502
예측 정확도: ['주변학교수', '주변병원수'] | 0.8448
예측 정확도: ['주변공원수', '주변학원수'] | 0.8529
예측 정확도: ['주변공원수', '주변병원수'] | 0.8466
예측 정확도: ['주변학원수', '주변병원수'] | 0.8546
예측 정확도: ['주변학교수', '주변공원수', '주변학원수'] | 0.8573
예측 정확도: ['주변학교수', '주변공원수', '주변병원수'] | 0.8561
예측 정확도: ['주변학교수', '주변학원수', '주변병원수'] | 0.8575
예측 정확도: ['주변공원수', '주변학원수', '주변병원수'] | 0.8598
예측 정확도: ['주변학교수', '주변공원수', '주변학원수', '주변병원수'] | 0.8594


In [ ]:
def calculate_shortest_distance(apt_df, infra_df):
    if infra_df.empty or '위도' not in infra_df.columns:
        return np.zeros(len(apt_df))
        
    infra_coords = infra_df[['위도', '경도']].values
    tree = cKDTree(infra_coords)
    
    apt_coords = apt_df[['위도', '경도']].values
    
    dists, _ = tree.query(apt_coords, k=1)
    
    distance_meters = dists * 111000
    return distance_meters

df['최단학교거리'] = calculate_shortest_distance(df, school)
df['최단공원거리'] = calculate_shortest_distance(df, park)
df['최단학원거리'] = calculate_shortest_distance(df, academy)
df['최단병원거리'] = calculate_shortest_distance(df, hospital)

print(f'최종 데이터셋 크기(Shape): {df.shape}')

최종 데이터셋 크기(Shape): (173399, 23)


In [ ]:
distance_features = ['최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리']

possible_base = ['전용면적', '층', '건축년도', '건물면적', '층정보']
base_features = [col for col in possible_base if col in df.columns]

target_col = None
for col in ['실거래가', '물건금액', '물건금액(만원)', '가격', '거래금액', 'price']:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    potential_cols = [c for c in df.columns if '금액' in c or '가' in c or 'price' in c.lower()]
    target_col = potential_cols[0] if potential_cols else df.columns[-1]

y = df[target_col]
results = []

for k in range(1, len(distance_features) + 1):
    for comb in itertools.combinations(distance_features, k):
        selected_distance = list(comb)
        
        X_features = base_features + selected_distance
        X = df[X_features]
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        model = RandomForestRegressor(n_estimators=20, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        score = r2_score(y_test, y_pred)
        
        results.append({
            '개수': k, 
            '변수조합': selected_distance, 
            'R2_Score': score
        })
        
        print(f"예측 정확도: {selected_distance} | {score:.4f}")


예측 정확도: ['최단학교거리'] | 0.8429
예측 정확도: ['최단공원거리'] | 0.8388
예측 정확도: ['최단학원거리'] | 0.8428
예측 정확도: ['최단병원거리'] | 0.8405
예측 정확도: ['최단학교거리', '최단공원거리'] | 0.8490
예측 정확도: ['최단학교거리', '최단학원거리'] | 0.8498
예측 정확도: ['최단학교거리', '최단병원거리'] | 0.8499
예측 정확도: ['최단공원거리', '최단학원거리'] | 0.8465
예측 정확도: ['최단공원거리', '최단병원거리'] | 0.8480
예측 정확도: ['최단학원거리', '최단병원거리'] | 0.8477
예측 정확도: ['최단학교거리', '최단공원거리', '최단학원거리'] | 0.8518
예측 정확도: ['최단학교거리', '최단공원거리', '최단병원거리'] | 0.8502
예측 정확도: ['최단학교거리', '최단학원거리', '최단병원거리'] | 0.8516
예측 정확도: ['최단공원거리', '최단학원거리', '최단병원거리'] | 0.8491
예측 정확도: ['최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리'] | 0.8526


In [109]:


elite_features = [
    '주변공원수', '주변학원수', '주변병원수',
    '최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리'
]

possible_base = ['전용면적', '층', '건축년도', '건물면적', '층정보']
base_features = [col for col in possible_base if col in df.columns]

X_features = base_features + elite_features

target_col = None
for col in ['실거래가', '물건금액', '물건금액(만원)', '가격', '거래금액', 'price']:
    if col in df.columns:
        target_col = col
        break
if target_col is None:
    potential_cols = [c for c in df.columns if '금액' in c or '가' in c or 'price' in c.lower()]
    target_col = potential_cols[0] if potential_cols else df.columns[-1]

X = df[X_features]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = r2_score(y_test, y_pred)
print("['주변공원수', '주변학원수', '주변병원수', '최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리']")
print(f"예측 정확도: {score:.4f}")

['주변공원수', '주변학원수', '주변병원수', '최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리']
예측 정확도: 0.8592


In [110]:


elite_features = [
    '주변공원수', '주변학원수', '주변병원수',
    '최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리'
]

possible_base = ['전용면적', '층', '건축년도', '건물면적', '층정보']
base_features = [col for col in possible_base if col in df.columns]

X_features = base_features + elite_features

target_col = None
for col in ['실거래가', '물건금액', '물건금액(만원)', '가격', '거래금액', 'price']:
    if col in df.columns:
        target_col = col
        break
if target_col is None:
    potential_cols = [c for c in df.columns if '금액' in c or '가' in c or 'price' in c.lower()]
    target_col = potential_cols[0] if potential_cols else df.columns[-1]

X = df[X_features]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = r2_score(y_test, y_pred)

print("결정트리 개수 변경 50")
print("['주변공원수', '주변학원수', '주변병원수', '최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리']")
print(f"예측 정확도 : {score:.4f}")

결정트리 개수 변경 50
['주변공원수', '주변학원수', '주변병원수', '최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리']
예측 정확도 : 0.8598


In [111]:


optimized_features = [
    '주변학원수', '주변병원수',        
    '최단학교거리', '최단공원거리', '최단병원거리' 
]

possible_base = ['전용면적', '층', '건축년도', '건물면적', '층정보']
base_features = [col for col in possible_base if col in df.columns]

X_features = base_features + optimized_features

target_col = None
for col in ['실거래가', '물건금액', '물건금액(만원)', '가격', '거래금액', 'price']:
    if col in df.columns:
        target_col = col
        break
if target_col is None:
    potential_cols = [c for c in df.columns if '금액' in c or '가' in c or 'price' in c.lower()]
    target_col = potential_cols[0] if potential_cols else df.columns[-1]

X = df[X_features]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = r2_score(y_test, y_pred)

print("['주변학원수', '주변병원수', '최단학교거리', '최단공원거리', '최단병원거리' ]")
print(f"예측 정확도 : {score:.4f}")

['주변학원수', '주변병원수', '최단학교거리', '최단공원거리', '최단병원거리' ]
예측 정확도 : 0.8597


In [112]:


elite_features = ['주변공원수', '주변학원수', '주변병원수', '최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리']

possible_base = ['전용면적', '층', '건축년도', '건물면적', '층정보', '계약년', '계약월']
base_features = [col for col in possible_base if col in df.columns]

X_features = base_features + elite_features

target_col = None
for col in ['실거래가', '물건금액', '물건금액(만원)', '가격', '거래금액', 'price']:
    if col in df.columns:
        target_col = col
        break
if target_col is None:
    potential_cols = [c for c in df.columns if '금액' in c or '가' in c or 'price' in c.lower()]
    target_col = potential_cols[0] if potential_cols else df.columns[-1]

X = df[X_features]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = r2_score(y_test, y_pred)

print("['주변공원수', '주변학원수', '주변병원수', '최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리']")
print(f"예측 정확도: {score:.4f}")

['주변공원수', '주변학원수', '주변병원수', '최단학교거리', '최단공원거리', '최단학원거리', '최단병원거리']
예측 정확도: 0.7285


## 5. EDA — 변수 분석

AI 활용 자유롭게.

- 변수 분포 (히스토그램, 박스플롯)
- 거래금액과의 산점도/상관관계
- 강남/비강남 비교
- 구간별 평균 가격
- 다른 변수와의 다중공선성

In [ ]:
from utils import nearest_distance, count_within_radius

def add_features_unique(df, target_df, prefix, radius=1000,
                        key_cols=['단지명', '도로명'],
                        lat_col='위도', lon_col='경도'):
    dist_col = f'{prefix}_nearest_dist'
    count_col = f'{prefix}_count_{radius}m'

    unique_apt = df[key_cols + ['위도', '경도']].drop_duplicates().reset_index(drop=True)

    unique_apt[dist_col] = unique_apt.apply(
        lambda row: nearest_distance(
            row['위도'], row['경도'], target_df, lat_col=lat_col, lon_col=lon_col
        ),
        axis=1
    )
    unique_apt[count_col] = unique_apt.apply(
        lambda row: count_within_radius(
            row['위도'], row['경도'], target_df, radius=radius,
            lat_col=lat_col, lon_col=lon_col
        ),
        axis=1
    )

    df = df.merge(unique_apt[key_cols + [dist_col, count_col]], on=key_cols, how='left')
    return df

subway_df = pd.read_csv('../data/raw/서울지하철.csv')
bus_df = pd.read_csv('../data/raw/서울버스정류장.csv', encoding='cp949')

def add_transport_features(df, subway_df=None, bus_df=None,
                           subway_radius=500, bus_radius=500):
    if subway_df is not None:
        subway_df = subway_df.rename(columns={'역위도': '위도', '역경도': '경도'})
        df = add_features_unique(df, subway_df, 'subway', radius=subway_radius)

    if bus_df is not None:
        df = add_features_unique(df, bus_df, 'bus', radius=bus_radius)
        bus_count_col = f'bus_count_{bus_radius}m'
        if bus_count_col in df.columns:
            df = df.drop(columns=[bus_count_col])
    return df

if not ('subway_nearest_dist' in df.columns and 'bus_nearest_dist' in df.columns):
    df = add_transport_features(df, subway_df=subway_df, bus_df=bus_df, subway_radius=500, bus_radius=500)

traffic_features = ['subway_nearest_dist', 'subway_count_500m', 'bus_nearest_dist']
possible_base = ['전용면적', '전용면적(㎡)', '층', '건축년도', '노후도', '건물면적']
base_features = [col for col in possible_base if col in df.columns]

X_features = base_features + traffic_features

target_col = None
for col in df.columns:
    if any(keyword in col for keyword in ['실거래가', '물건금액', '거래금액', '가격', '금액']):
        if not col.endswith('_x') and not col.endswith('_y'):
            target_col = col
            break

if target_col is None:
    raise ValueError(f"Target column not found. Available columns: {df.columns.tolist()}")

all_cols_to_use = X_features + [target_col]
clean_df = df[all_cols_to_use].dropna().copy()

X = clean_df[X_features]
y = clean_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = r2_score(y_test, y_pred)

print("변수 조합 : 가장 가까운 지하철역까지의 거리, 아파트 반경 500m 이내에 있는 지하철역의 개수, 가장 가까운 버스정류장까지의 거리")
print(f"예측 정확도: {score:.4f}")

변수 조합 : 가장 가까운 지하철역까지의 거리, 아파트 반경 500m 이내에 있는 지하철역의 개수, 가장 가까운 버스정류장까지의 거리
예측 정확도: 0.9510


In [114]:

jonguk_features = ['주변공원수', '주변학원수', '주변병원수']
chaeyoung_features = ['subway_nearest_dist', 'subway_count_500m', 'bus_nearest_dist']

possible_base = ['전용면적', '전용면적(㎡)', '층', '건축년도', '노후도', '건물면적']
base_features = [col for col in possible_base if col in df.columns]

X_features = base_features + jonguk_features + chaeyoung_features

target_col = None
for col in df.columns:
    if any(keyword in col for keyword in ['실거래가', '물건금액', '거래금액', '가격', '금액']):
        if not col.endswith('_x') and not col.endswith('_y'):
            target_col = col
            break

if target_col is None:
    raise ValueError(f"Target column not found. Available columns: {df.columns.tolist()}")

all_cols_to_use = X_features + [target_col]
clean_df = df[all_cols_to_use].dropna().copy()

X = clean_df[X_features]
y = clean_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = r2_score(y_test, y_pred)

print("변수 조합 : 주변공원수, 주변학원수, 주변병원수, 최근접 지하철역 거리, 500m 내 지하철역 개수, 최근접 버스정류장 거리")
print(f"예측 정확도: {score:.4f}")

변수 조합 : 주변공원수, 주변학원수, 주변병원수, 최근접 지하철역 거리, 500m 내 지하철역 개수, 최근접 버스정류장 거리
예측 정확도: 0.9598


## 6. 채택/탈락 판단

분석 결과를 보고 어떤 변수를 채택할지 결정.

**참고:** 채택/탈락 근거는 자기 작업 노트(워드/노션/메모장 등)에 자유 양식으로 기록.
보고서 작성 시 그 노트를 가져와서 보고서에 옮긴다.

## 7. features.py에 함수로 옮기기

분석 결과를 토대로 채택한 변수만 함수로 정리하여 `src/features.py`에 추가.

예시:
```python
# src/features.py — 본인 담당 섹션에 추가

def add_my_features(df, my_df, radius=1000):
    """
    [내 변수 그룹] 변수 추가.

    추가 컬럼:
        - prefix_nearest_dist
        - prefix_count_{r}m
    """
    df = add_features_unique(df, my_df, 'prefix', radius=radius)
    return df
```

함수 시그니처 규약은 `src/features.py` 상단 주석 참조.